In [ ]:
import os
import sys
import numpy as np
import warnings
import tensorflow as tf

tf.get_logger().setLevel("ERROR")
tf.autograph.set_verbosity(0)

ROOT = os.path.abspath("..")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import run_unet_mlp_uv15 as base
import run_unet_mlp_bottleneckvqc_uv15_randomsplit as exp

from functions.nb_helpers import (
    signed_log1p,
    transform_y_signed_log1p,
    repo_root,
    resolve_extracted_uv_dir,
    resolve_ood_w07_120_path,
    evaluate_split_physical,
    plot_uv_threeway,
)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
# Config
height_m = 15.0
last_k = 12
epochs = 1000
batch_size = 2
cond_emb_dim = 16
n_qubits = 5
n_layers = 2
seed = 7

# train/val split within extracted_uv (40 cases) only
train_frac = 0.8
val_frac = 0.1

base.set_seeds(seed)

In [ ]:
here = str(repo_root())
os.chdir(here)

extracted_uv_dir = str(resolve_extracted_uv_dir(here))
ood_path = str(resolve_ood_w07_120_path(here))

# Main grid for train/val (skips *_deg* OOD files by default)
train_val_cases = base.list_cases(extracted_uv_dir)
# Single OOD extrapolation case: 7 m/s, 120° under extracted_uv/
test_cases = [base.parse_case_from_filename(ood_path)]

if len(test_cases) != 1:
    raise ValueError(f"Expected exactly 1 OOD test case, got {len(test_cases)}")
if int(test_cases[0].speed) != 7 or abs(float(test_cases[0].angle_deg) - 120.0) > 1e-6:
    raise ValueError(
        f"Expected OOD case 7 m/s @ 120°, got speed={test_cases[0].speed}, "
        f"angle={test_cases[0].angle_deg} (file={os.path.basename(test_cases[0].path)})"
    )

print(f"train/val candidates (extracted_uv): {len(train_val_cases)}")
print(f"OOD test cases: {len(test_cases)}")
print(
    f"OOD extrapolation case: speed={test_cases[0].speed} m/s, "
    f"angle={test_cases[0].angle_deg}°, "
    f"file={os.path.basename(test_cases[0].path)}"
)

cases = train_val_cases + test_cases
n_train_val = len(train_val_cases)

xs, cs, ys, meta = [], [], [], []
for i, c in enumerate(cases):
    m_building, u_mean, v_mean = base.load_uv_steady_mean(c.path, height_m=height_m, last_k=last_k)
    x = m_building[..., None].astype(np.float32)
    y = np.stack([u_mean, v_mean], axis=-1).astype(np.float32)
    angle_deg = float(c.angle_deg)

    xs.append(x)
    cs.append(base.cond_vector(c.speed, angle_deg))
    ys.append(y)
    meta.append({
        "file": os.path.basename(c.path),
        "speed": c.speed,
        "d_code": c.d_code,
        "angle_deg": angle_deg,
        "split_group": "train_val" if i < n_train_val else "ood_test",
    })

X = np.stack(xs, axis=0)
C = np.stack(cs, axis=0)
Y = np.stack(ys, axis=0)

X.shape, C.shape, Y.shape


In [ ]:
# Non-building u_mean / v_mean distributions
u_all = Y[..., 0]
v_all = Y[..., 1]

valid_mask = (u_all != base.MISSING_VALUE) & (v_all != base.MISSING_VALUE)
u_valid = u_all[valid_mask]
v_valid = v_all[valid_mask]

# Signed log1p transform: x -> sign(x) * log1p(|x|)

u_log = signed_log1p(u_valid)
v_log = signed_log1p(v_valid)

# Train targets in signed-log1p space (buildings keep missing value)
Y_log = transform_y_signed_log1p(Y, base.MISSING_VALUE)


In [ ]:
# Split scheme:
# - train/val: extracted_uv (40 known speed/direction cases)
# - test: single OOD case extracted_uv/extracted_w07_deg120_3d.000.nc (7 m/s, 120°)

if train_frac + val_frac >= 1.0:
    raise ValueError(f"train_frac+val_frac must be < 1. Got {train_frac+val_frac}")

train_val_idx = [i for i, m in enumerate(meta) if m["split_group"] == "train_val"]
test_idx = [i for i, m in enumerate(meta) if m["split_group"] == "ood_test"]

if len(train_val_idx) == 0:
    raise ValueError("train/val is empty; check extracted_uv")
if len(test_idx) == 0:
    raise ValueError("test is empty; check extracted_uv/extracted_w07_deg120_3d.000.nc")

# Stratified val sampling by wind speed within extracted_uv
rng = np.random.default_rng(seed)
val_idx = []
for s in sorted({meta[i]["speed"] for i in train_val_idx}):
    idx_s = [i for i in train_val_idx if meta[i]["speed"] == s]
    rng.shuffle(idx_s)
    n_val_s = max(1, int(round(len(idx_s) * val_frac)))
    val_idx.extend(idx_s[:n_val_s])

val_idx = sorted(set(val_idx))
train_idx = sorted(set(train_val_idx) - set(val_idx))
test_idx = sorted(test_idx)

if not train_idx:
    raise ValueError("train is empty; reduce val_frac")

covered = set(train_idx) | set(val_idx) | set(test_idx)
all_idx = set(range(len(cases)))
if covered != all_idx:
    missing = sorted(all_idx - covered)
    overlap = len(train_idx) + len(val_idx) + len(test_idx) - len(covered)
    raise ValueError(f"Invalid split: missing={missing}, overlap_count={overlap}")

split_idx = {"train": train_idx, "val": val_idx, "test": test_idx}
tr, va, te = split_idx["train"], split_idx["val"], split_idx["test"]

print(f"split sizes => train: {len(tr)}, val: {len(va)}, test: {len(te)}")

# Condition scaling (fit on train only)
c_scaler = base.StandardScaler()
C_tr = c_scaler.fit_transform(C[tr])
C_va = c_scaler.transform(C[va])
C_te = c_scaler.transform(C[te])

# Output normalization in signed-log1p space (fit on train only, ignoring missing)
y_mean, y_std = base.compute_y_norm_stats(Y_log[tr])
Y_tr = base.normalize_y(Y_log[tr], y_mean, y_std)
Y_va = base.normalize_y(Y_log[va], y_mean, y_std)
Y_te = base.normalize_y(Y_log[te], y_mean, y_std)

X_tr, X_va, X_te = X[tr], X[va], X[te]
len(tr), len(va), len(te)


# C-QB-UNet

In [ ]:
# # Full epochs training
# model = exp.build_unet_cond_mlp_bottleneck_vqc(
#     input_shape=X_tr.shape[1:],
#     cond_dim=C_tr.shape[-1],
#     cond_emb_dim=cond_emb_dim,
#     n_qubits=n_qubits,
#     n_layers=n_layers,
# )

# model.compile(
#     optimizer=base.tf.keras.optimizers.Adam(1e-3),
#     loss=base.masked_mse_with_grad,
#     metrics=[base.masked_mae_metric, base.masked_rmse_metric, base.MaskedR2()],
# )

# checkpoint_dir = os.path.join(here, "checkpoints", "bottleneckvqc_unet_log1p_w07")
# os.makedirs(checkpoint_dir, exist_ok=True)
# best_weights_path = os.path.join(checkpoint_dir, "best_val_weights.h5")

# callbacks = [
#     base.tf.keras.callbacks.ModelCheckpoint(
#         filepath=best_weights_path,
#         monitor="val_loss",
#         save_best_only=True,
#         save_weights_only=True,
#         mode="min",
#         verbose=1,
#     ),
#     base.tf.keras.callbacks.ReduceLROnPlateau(
#         monitor="val_loss", factor=0.5, patience=10, min_lr=1e-5, verbose=1
#     ),
# ]

# history = model.fit(
#     x={"mask_img": X_tr, "cond": C_tr},
#     y=Y_tr,
#     validation_data=({"mask_img": X_va, "cond": C_va}, Y_va),
#     epochs=epochs,
#     batch_size=batch_size,
#     callbacks=callbacks,
#     verbose=1,
# )

# # Load best val_loss weights after training
# model.load_weights(best_weights_path)


In [ ]:
# load pretrained weights
save_dir = os.path.join(here, "checkpoints", "bottleneckvqc_unet_log1p")
os.makedirs(save_dir, exist_ok=True)
best_weights_path = os.path.join(save_dir, "best_val_weights_w07.h5")

# Reload model and stats (rebuild the same architecture, then load weights)
model = exp.build_unet_cond_mlp_bottleneck_vqc(
    input_shape=X_tr.shape[1:],
    cond_dim=C_tr.shape[-1],
    cond_emb_dim=cond_emb_dim,
    n_qubits=n_qubits,
    n_layers=n_layers,
)

model.compile(
    optimizer=base.tf.keras.optimizers.Adam(1e-3),
    loss=base.masked_mse_with_grad,
    metrics=[base.masked_mae_metric, base.masked_rmse_metric, base.MaskedR2()],
)

model.load_weights(best_weights_path)

print("Train/Val/Test metrics (evaluated in m/s after inverse transform):")

Y_tr_denorm, pred_tr_denorm, metrics_tr = evaluate_split_physical(model, "train", X_tr, C_tr, Y_tr, y_mean, y_std)
Y_va_denorm, pred_va_denorm, metrics_va = evaluate_split_physical(model, "val", X_va, C_va, Y_va, y_mean, y_std)
Y_te_denorm, pred_te_denorm, metrics_te = evaluate_split_physical(model, "test", X_te, C_te, Y_te, y_mean, y_std)

# C-UNet

In [ ]:
# # Full epochs training
# # C-UNet config (reuse same data/split)
# mlp_epochs = epochs
# mlp_batch_size = batch_size

# model_mlp = base.build_unet_cond(input_shape=X_tr.shape[1:], cond_dim=C_tr.shape[-1])
# model_mlp.compile(
#     optimizer=base.tf.keras.optimizers.Adam(1e-3),
#     loss=base.masked_mse_with_grad,
#     metrics=[base.masked_mae_metric, base.masked_rmse_metric, base.MaskedR2()],
# )

# checkpoint_dir = os.path.join(here, "checkpoints", "mlp_unet_log1p_w07")
# os.makedirs(checkpoint_dir, exist_ok=True)
# best_weights_path = os.path.join(checkpoint_dir, "best_val_weights.h5")

# callbacks_mlp = [
#     base.tf.keras.callbacks.ModelCheckpoint(
#         filepath=best_weights_path,
#         monitor="val_loss",
#         save_best_only=True,
#         save_weights_only=True,
#         mode="min",
#         verbose=1,
#     ),
#     base.tf.keras.callbacks.ReduceLROnPlateau(
#         monitor="val_loss", factor=0.5, patience=10, min_lr=1e-5, verbose=1
#     ),
# ]

# history_mlp = model_mlp.fit(
#     x={"mask_img": X_tr, "cond": C_tr},
#     y=Y_tr,
#     validation_data=({"mask_img": X_va, "cond": C_va}, Y_va),
#     epochs=mlp_epochs,
#     batch_size=mlp_batch_size,
#     callbacks=callbacks_mlp,
#     verbose=1,
# )

# # Load best val_loss weights after training
# model_mlp.load_weights(best_weights_path)


In [ ]:
# load pretrained weights
save_dir = os.path.join(here, "checkpoints", "mlp_unet_log1p")
os.makedirs(save_dir, exist_ok=True)
best_weights_path = os.path.join(save_dir, "best_val_weights_w07.h5")

# Reload model and stats (rebuild the same architecture, then load weights)
model_mlp = base.build_unet_cond(input_shape=X_tr.shape[1:], cond_dim=C_tr.shape[-1])

model_mlp.compile(
    optimizer=base.tf.keras.optimizers.Adam(1e-3),
    loss=base.masked_mse_with_grad,
    metrics=[base.masked_mae_metric, base.masked_rmse_metric, base.MaskedR2()],
)

model_mlp.load_weights(best_weights_path)

print("C-UNet Train/Val/Test metrics (evaluated in m/s after inverse transform):")

Y_tr_denorm_mlp, pred_tr_denorm_mlp, metrics_tr_mlp = evaluate_split_physical(model_mlp, "train", X_tr, C_tr, Y_tr, y_mean, y_std)
Y_va_denorm_mlp, pred_va_denorm_mlp, metrics_va_mlp = evaluate_split_physical(model_mlp, "val", X_va, C_va, Y_va, y_mean, y_std)
Y_te_denorm_mlp, pred_te_denorm_mlp, metrics_te_mlp = evaluate_split_physical(model_mlp, "test", X_te, C_te, Y_te, y_mean, y_std)

# U/V Spatial Field

In [ ]:
show_ids = [0]  # OOD test has a single case: 7 m/s, 120°
for i in show_ids:
    m = meta[te[i]]
    plot_uv_threeway(
        Y_te_denorm,
        pred_te_denorm,
        pred_te_denorm_mlp,
        i,
        title=f"Test U/V Field Comparison ({m['speed']}m/s {m['angle_deg']}deg)",
    )


In [ ]:
# locate sample: 6 m/s, 135° (truth + two models)

target_speed = 6
target_angle = 135.0
tol = 1e-6

# map global index to split-specific local index
tr_map = {int(g): i for i, g in enumerate(tr)}
va_map = {int(g): i for i, g in enumerate(va)}
te_map = {int(g): i for i, g in enumerate(te)}

cands = [
    i for i, m in enumerate(meta)
    if int(m["speed"]) == target_speed and abs(float(m["angle_deg"]) - target_angle) < tol
]

if len(cands) == 0:
    raise ValueError(f"No sample found for speed={target_speed}, angle={target_angle}")

priority = []
for gi in cands:
    if gi in tr_map:
        priority.append((0, gi))
    elif gi in va_map:
        priority.append((1, gi))
    elif gi in te_map:
        priority.append((2, gi))
    else:
        priority.append((3, gi))

priority.sort()
gidx = priority[0][1]
m = meta[gidx]

if gidx in tr_map:
    split_name = "train"
    li = tr_map[gidx]
    y_true_split = Y_tr_denorm
    y_vqc_split = pred_tr_denorm
    y_mlp_split = pred_tr_denorm_mlp
elif gidx in va_map:
    split_name = "val"
    li = va_map[gidx]
    y_true_split = Y_va_denorm
    y_vqc_split = pred_va_denorm
    y_mlp_split = pred_va_denorm_mlp
elif gidx in te_map:
    split_name = "test"
    li = te_map[gidx]
    y_true_split = Y_te_denorm
    y_vqc_split = pred_te_denorm
    y_mlp_split = pred_te_denorm_mlp

print(
    f"selected sample -> split={split_name}, global_idx={gidx}, local_idx={li}, "
    f"file={m['file']}, speed={m['speed']}, angle={m['angle_deg']}"
)

# three-way comparison (True / MLP-UNet / MLP-BottleneckVQC)
plot_uv_threeway(
    y_true_split,
    y_vqc_split,
    y_mlp_split,
    li,
    title=(f"Train U/V Field Comparison (6m/s 135deg)"),
)